In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 500) 

In [ ]:
import sys
import os

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_path)

from app.model_download import CACHE_DIR
from app.settings import settings

TEST_AUDIO_FILE = settings.AUDIO_DIR / "test_video_3.mp4"
CACHE_DIR = settings.CACHE_DIR

MODEL_NAME = "pyannote/speaker-diarization-community-1"
# pyannote/speaker-diarization-3.1

In [ ]:
from pydub import AudioSegment

def get_audio_duration(path: str) -> float:
    audio = AudioSegment.from_file(path)
    return len(audio) / 1000.0

print(f"Audio duration: {get_audio_duration(str(TEST_AUDIO_FILE))} seconds")

In [ ]:
import torch
import torchaudio
from pyannote.audio import Pipeline


def load_pipeline():
    """
    Инициализация diarization pipeline с кэшированием и выбором устройства
    """
    pipeline = Pipeline.from_pretrained(
        MODEL_NAME,
        token=settings.HF_TOKEN,
        cache_dir=CACHE_DIR
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pipeline.to(device)

    return pipeline


def load_audio(audio_path):
    """
    Загрузка аудио в память (быстрее, чем передача пути)
    """
    waveform, sample_rate = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    return {
        "waveform": waveform,
        "sample_rate": sample_rate
    }


from typing import Any


def run_diarization(pipeline, audio_data, use_exclusive: bool = False):
    output = pipeline(audio_data)

    max_end: float = 0.0
    speakers: set[str] = set()
    segments: list[dict[str, Any]] = []

    # --- основной diarization ---
    diarization = getattr(output, "speaker_diarization", None)

    if diarization is not None:
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            start = float(turn.start)
            end = float(turn.end)

            max_end = max(max_end, end)
            speakers.add(str(speaker))

            segments.append(
                {
                    "speaker": str(speaker),
                    "start": start,
                    "end": end,
                    "duration": end - start,
                }
            )

    # --- exclusive diarization ---
    exclusive_segments: list[dict[str, Any]] = []
    exclusive = getattr(output, "exclusive_speaker_diarization", None)

    if use_exclusive and exclusive is not None:
        for turn, _, speaker in exclusive.itertracks(yield_label=True):
            start = float(turn.start)
            end = float(turn.end)

            exclusive_segments.append(
                {
                    "speaker": str(speaker),
                    "start": start,
                    "end": end,
                    "duration": end - start,
                }
            )

    return {
        "segments": segments,
        "exclusive_segments": exclusive_segments,
        "speakers": list(speakers),
        "num_speakers": len(speakers),
        "duration": max_end,
        "raw": output,
    }


In [ ]:
import torch
import torchaudio
from pyannote.audio import Pipeline


def load_pipeline():
    """
    Инициализация diarization pipeline с кэшированием и выбором устройства
    """
    pipeline = Pipeline.from_pretrained(
        MODEL_NAME,
        token=settings.HF_TOKEN,
        cache_dir=CACHE_DIR
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pipeline.to(device)

    return pipeline


def load_audio(audio_path):
    """
    Загрузка аудио в память (быстрее, чем передача пути)
    """
    waveform, sample_rate = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    return {
        "waveform": waveform,
        "sample_rate": sample_rate
    }


from typing import Any


def run_diarization(pipeline, audio_data, use_exclusive: bool = False):
    output = pipeline(audio_data)

    max_end: float = 0.0
    speakers: set[str] = set()
    segments: list[dict[str, Any]] = []

    # --- основной diarization ---
    diarization = getattr(output, "speaker_diarization", None)

    if diarization is not None:
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            start = float(turn.start)
            end = float(turn.end)

            max_end = max(max_end, end)
            speakers.add(str(speaker))

            segments.append(
                {
                    "speaker": str(speaker),
                    "start": start,
                    "end": end,
                    "duration": end - start,
                }
            )

    # --- exclusive diarization ---
    exclusive_segments: list[dict[str, Any]] = []
    exclusive = getattr(output, "exclusive_speaker_diarization", None)

    if use_exclusive and exclusive is not None:
        for turn, _, speaker in exclusive.itertracks(yield_label=True):
            start = float(turn.start)
            end = float(turn.end)

            exclusive_segments.append(
                {
                    "speaker": str(speaker),
                    "start": start,
                    "end": end,
                    "duration": end - start,
                }
            )

    return {
        "segments": segments,
        "exclusive_segments": exclusive_segments,
        "speakers": list(speakers),
        "num_speakers": len(speakers),
        "duration": max_end,
        "raw": output,
    }


In [ ]:
pipeline = load_pipeline()
audio_data = load_audio(TEST_AUDIO_FILE)

result = run_diarization(
    pipeline,
    audio_data,
    use_exclusive=True
)
result